# Week 76, Day 3: Calling APIs

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('fakeapi.py', 'w', encoding='utf-8') as f:
    f.write('"""A pretend web API that runs in the page, for practising API clients (Week 76).\n\napi = FakeAPI() makes a fresh API. Call it like a transport:\n    response = api("GET", "https://api.example.com/books?page=1", headers={...}, body=None, timeout=5)\nIt returns a Response with .status, .headers (lower-case names), .body (bytes), .json(), .ok\nand .raise_for_status(), and records every request it receives in api.log.\n"""\nimport json\nfrom urllib.parse import urlsplit, parse_qs\n\nBASE = "https://api.example.com"\n\nBOOKS = [\n    {"id": 1, "title": "Emma", "author": "Austen"},\n    {"id": 2, "title": "Persuasion", "author": "Austen"},\n    {"id": 3, "title": "Bleak House", "author": "Dickens"},\n    {"id": 4, "title": "Middlemarch", "author": "Eliot"},\n    {"id": 5, "title": "Dracula", "author": "Stoker"},\n    {"id": 6, "title": "Frankenstein", "author": "Shelley"},\n    {"id": 7, "title": "Ivanhoe", "author": "Scott"},\n]\n\n\nclass HTTPError(Exception):\n    def __init__(self, response):\n        super().__init__(f"{response.status} error")\n        self.response = response\n\n\nclass Response:\n    def __init__(self, status, headers=None, body=b""):\n        self.status = status\n        self.headers = {k.lower(): v for k, v in (headers or {}).items()}\n        self.body = body\n\n    @property\n    def ok(self):\n        return self.status < 400\n\n    def json(self):\n        return json.loads(self.body.decode("utf-8"))\n\n    def raise_for_status(self):\n        if self.status >= 400:\n            raise HTTPError(self)\n\n    def __repr__(self):\n        return f"<Response {self.status}>"\n\n\ndef _json(status, data, headers=None):\n    return Response(status, {"Content-Type": "application/json", **(headers or {})}, json.dumps(data).encode("utf-8"))\n\n\nclass FakeAPI:\n    def __init__(self, token="test-token-123", page_size=3, fail_first=(), delay=0.0, limit=None):\n        self.books = [dict(b) for b in BOOKS]\n        self.token = token\n        self.page_size = page_size\n        self.fail_first = list(fail_first)     # statuses (or exception classes) to return before working normally\n        self.delay = delay                     # seconds each request "takes"\n        self.limit = limit                     # requests allowed before 429s start (None for no limit)\n        self.log = []\n\n    def __call__(self, method, url, headers=None, body=None, timeout=None):\n        headers = {k.lower(): v for k, v in (headers or {}).items()}\n        self.log.append((method.upper(), url))\n        if timeout is not None and self.delay > timeout:\n            raise TimeoutError(f"no response within {timeout} seconds")\n        if self.fail_first:\n            failure = self.fail_first.pop(0)\n            if isinstance(failure, type) and issubclass(failure, BaseException):\n                raise failure("connection failed")\n            return _json(failure, {"error": "try again"}, {"Retry-After": "1"} if failure in (429, 503) else None)\n        if self.limit is not None and len(self.log) > self.limit:\n            return _json(429, {"error": "slow down"}, {"Retry-After": "2"})\n        parts = urlsplit(url)\n        if f"{parts.scheme}://{parts.netloc}" != BASE:\n            raise ConnectionError(f"can\'t reach {parts.netloc}")\n        query = {k: v[0] for k, v in parse_qs(parts.query).items()}\n        path = parts.path.rstrip("/")\n        method = method.upper()\n        if path == "/books" and method == "GET":\n            return self._page(query)\n        if path == "/books/cursor" and method == "GET":\n            return self._cursor(query)\n        if path == "/books" and method == "POST":\n            return self._create(headers, body)\n        if path.startswith("/books/"):\n            tail = path[len("/books/"):]\n            if not tail.isdigit():\n                return _json(404, {"error": "not found"})\n            return self._one(method, int(tail), headers)\n        return _json(404, {"error": "not found"})\n\n    def _page(self, query):\n        try:\n            page = int(query.get("page", "1"))\n        except ValueError:\n            return _json(400, {"error": "page must be a number"})\n        size = self.page_size\n        pages = max(1, -(-len(self.books) // size))\n        if page < 1:\n            return _json(400, {"error": "page must be at least 1"})\n        items = self.books[(page - 1) * size: page * size]\n        return _json(200, {"items": items, "page": page, "pages": pages})\n\n    def _cursor(self, query):\n        after = int(query.get("after", "0"))\n        later = [b for b in self.books if b["id"] > after]\n        items = later[: self.page_size]\n        more = len(later) > self.page_size\n        return _json(200, {"items": items, "next": items[-1]["id"] if more else None})\n\n    def _authorised(self, headers):\n        return headers.get("authorization") == f"Bearer {self.token}"\n\n    def _create(self, headers, body):\n        if not self._authorised(headers):\n            return _json(401, {"error": "a valid token is required"}, {"WWW-Authenticate": "Bearer"})\n        try:\n            data = json.loads(body or b"{}")\n        except ValueError:\n            return _json(400, {"error": "the body must be JSON"})\n        missing = [f for f in ("title", "author") if not str(data.get(f, "")).strip()]\n        if missing:\n            return _json(422, {"error": f"missing: {\', \'.join(missing)}"})\n        book = {"id": max(b["id"] for b in self.books) + 1, "title": data["title"], "author": data["author"]}\n        self.books.append(book)\n        return _json(201, book, {"Location": f"/books/{book[\'id\']}"})\n\n    def _one(self, method, book_id, headers):\n        book = next((b for b in self.books if b["id"] == book_id), None)\n        if method == "GET":\n            return _json(200, book) if book else _json(404, {"error": "not found"})\n        if method == "DELETE":\n            if not self._authorised(headers):\n                return _json(401, {"error": "a valid token is required"})\n            if book:\n                self.books.remove(book)\n            return Response(204)\n        return _json(405, {"error": "method not allowed"}, {"Allow": "GET, DELETE"})\n')

**Predict before you run:** what will this print?



In [ ]:
from fakeapi import FakeAPI, BASE

api = FakeAPI()
response = api("GET", f"{BASE}/books/3", headers={"Accept": "application/json"}, timeout=5)
print(response.status, response.headers["content-type"])
book = response.json()
print(book["title"], "by", book["author"])
print(api("GET", f"{BASE}/books/99", timeout=5))

In [ ]:
from fakeapi import FakeAPI, BASE

api = FakeAPI(page_size=3)

def books(api):
    after = 0
    while True:
        response = api("GET", f"{BASE}/books/cursor?after={after}", timeout=5)
        response.raise_for_status()
        page = response.json()
        yield from page["items"]
        if page["next"] is None:
            return
        after = page["next"]

for book in books(api):
    print(book["id"], book["title"])
print(len(api.log), "requests")

api.log.clear()
first_two = [b["title"] for _, b in zip(range(2), books(api))]
print(first_two, len(api.log), "request")

In [ ]:
from fakeapi import FakeAPI, BASE

api = FakeAPI(fail_first=[503, ConnectionError, 502])
waits = []

def get_with_retries(api, url, attempts=5, base=0.5, sleep=waits.append):
    for attempt in range(attempts):
        try:
            response = api("GET", url, timeout=5)
        except (ConnectionError, TimeoutError) as e:
            print("attempt", attempt + 1, "failed:", type(e).__name__)
        else:
            if response.status not in (429, 502, 503, 504):
                return response
            print("attempt", attempt + 1, "got", response.status)
        sleep(base * 2 ** attempt)
    raise RuntimeError("gave up")

print(get_with_retries(api, f"{BASE}/books/1").json()["title"], "after waiting", waits)

> **Key idea.** Call JSON APIs with a method, URL, headers, body and always a timeout; params= encodes queries, json= encodes bodies; check the status before decoding (raise_for_status). Sessions keep settings and reuse connections; inject the transport so clients can be tested with fakes. Use 2xx bodies, treat 404 as "none" where it means that, don't retry 400/422, authenticate on 401; send tokens in Authorization headers, read from the environment, never in code or logs. Paginate by page number or, more robustly, by cursor, lazily with a generator. Retry only idempotent requests (or POSTs with an idempotency key) after network errors and 502/503/504, with exponential backoff and jitter; on 429, wait for Retry-After and pace requests to stay under rate limits.

### Exercise: One book, or none

Write `get_book(api, book_id)`: `GET {BASE}/books/{book_id}` with a 5-second timeout and an `Accept: application/json` header; return the book as a dict, `None` for a 404, and raise the response's `HTTPError` (with `raise_for_status()`) for any other error. Then `titles(api, ids)`: the titles of the books with those ids that exist, in the order given.

In [ ]:
from fakeapi import BASE

def get_book(api, book_id):
    ___

def titles(api, ids):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_getbook():\n    \"status codes as answers\"\n    from fakeapi import FakeAPI, HTTPError\n    api = FakeAPI()\n    assert get_book(api, 3)[\"title\"] == \"Bleak House\" and get_book(api, 99) is None\n    assert titles(api, [5, 42, 1]) == [\"Dracula\", \"Emma\"]\n    try:\n        get_book(FakeAPI(fail_first=[500]), 1)\n        assert False\n    except HTTPError as e:\n        assert e.response.status == 500\n    class Strict:\n        def __call__(self, method, url, headers=None, body=None, timeout=None):\n            assert timeout is not None, \"always set a timeout\"\n            assert (headers or {}).get(\"Accept\") == \"application/json\"\n            return FakeAPI()(method, url, headers, body, timeout)\n    get_book(Strict(), 1)")

### Exercise: Every page

Write `all_books(api)`: fetch `GET {BASE}/books?page=N` for `N` from 1, collecting the items, until you've fetched the page the response calls the last (`"pages"`); return the list. Build the query with `urlencode`. Then `count_requests(api)`: how many requests `all_books` made, from `api.log`.

In [ ]:
from urllib.parse import urlencode
from fakeapi import BASE

def all_books(api):
    ___

def count_requests(api):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pages():\n    \"page numbers\"\n    from fakeapi import FakeAPI\n    api = FakeAPI(page_size=3)\n    books = all_books(api)\n    assert [b[\"id\"] for b in books] == [1, 2, 3, 4, 5, 6, 7] and count_requests(api) == 3\n    api2 = FakeAPI(page_size=10)\n    assert len(all_books(api2)) == 7 and count_requests(api2) == 1")

### Exercise: Lazily, by cursor

Write a generator `iter_books(api, author=None)`: yield books from `GET {BASE}/books/cursor?after=ID`, following each response's `"next"` until it's `None`, fetching a page only when the previous one has been used up, and skipping books by other authors if `author` is given. Then `first(api, n, author=None)`: a list of the first `n` books from it (with `itertools.islice`), which must make no more requests than needed.

In [ ]:
import itertools
from fakeapi import BASE

def iter_books(api, author=None):
    ___

def first(api, n, author=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cursor():\n    \"a lazy, paginated generator\"\n    import types\n    from fakeapi import FakeAPI\n    api = FakeAPI(page_size=3)\n    gen = iter_books(api)\n    assert isinstance(gen, types.GeneratorType) and api.log == [], \"nothing is fetched until it's needed\"\n    assert [b[\"id\"] for b in gen] == [1, 2, 3, 4, 5, 6, 7] and len(api.log) == 3\n    api.log.clear()\n    assert [b[\"title\"] for b in first(api, 2)] == [\"Emma\", \"Persuasion\"] and len(api.log) == 1\n    api.log.clear()\n    assert [b[\"title\"] for b in first(api, 5, author=\"Austen\")] == [\"Emma\", \"Persuasion\"] and len(api.log) == 3")

### Exercise: Creating, with a token

Write `create_book(api, env, title, author)`: `POST {BASE}/books` with a JSON body, `Content-Type: application/json`, and `Authorization: Bearer TOKEN`, where the token comes from `env["BOOKS_TOKEN"]` (`env` stands for `os.environ`; raise `RuntimeError("BOOKS_TOKEN is not set")` if it's missing). Return `(book, location)` on `201`. On `401` raise `PermissionError` with the API's error message; on `422`, raise `ValueError` with the API's error message; anything else unexpected, raise with `raise_for_status()`.

In [ ]:
import json
from fakeapi import BASE

def create_book(api, env, title, author):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_create():\n    \"authentication and validation errors\"\n    from fakeapi import FakeAPI\n    api = FakeAPI(token=\"t-42\")\n    book, location = create_book(api, {\"BOOKS_TOKEN\": \"t-42\"}, \"Cranford\", \"Gaskell\")\n    assert book == {\"id\": 8, \"title\": \"Cranford\", \"author\": \"Gaskell\"} and location == \"/books/8\"\n    for env, args, exc, message in [\n        ({\"BOOKS_TOKEN\": \"wrong\"}, (\"X\", \"Y\"), PermissionError, \"a valid token is required\"),\n        ({\"BOOKS_TOKEN\": \"t-42\"}, (\"\", \"Y\"), ValueError, \"missing: title\"),\n        ({}, (\"X\", \"Y\"), RuntimeError, \"BOOKS_TOKEN is not set\"),\n    ]:\n        try:\n            create_book(api, env, *args)\n            assert False\n        except exc as e:\n            assert str(e) == message\n    assert \"t-42\" not in source")

### Exercise: Retrying safely

Write `send(api, method, url, attempts=4, base=1.0, sleep=None, headers=None, body=None)`: call `api(...)` with a 5-second timeout, retrying up to `attempts` times in all. Retry after `ConnectionError` or `TimeoutError` and after `502`, `503` and `504` responses **only for idempotent methods** (`GET`, `HEAD`, `OPTIONS`, `PUT`, `DELETE`); retry after `429` for any method. Before each retry, call `sleep(seconds)`: the `Retry-After` header's value (a number of seconds) if the response had one, otherwise `base * 2 ** (attempt - 1)` for the `attempt`-th retry. Return the final response; if the last attempt raised, re-raise its exception.

In [ ]:
IDEMPOTENT = {"GET", "HEAD", "OPTIONS", "PUT", "DELETE"}

def send(api, method, url, attempts=4, base=1.0, sleep=None, headers=None, body=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"backoff, Retry-After and idempotency\"\n    from fakeapi import FakeAPI, BASE\n    waits = []\n    api = FakeAPI(fail_first=[ConnectionError, 502, 503])\n    r = send(api, \"GET\", f\"{BASE}/books/1\", sleep=waits.append)\n    assert r.status == 200 and waits == [1.0, 2.0, 1.0] and len(api.log) == 4\n    waits.clear()\n    api = FakeAPI(fail_first=[502])\n    assert send(api, \"POST\", f\"{BASE}/books\", sleep=waits.append, body=b\"{}\").status == 502 and waits == [] and len(api.log) == 1\n    api = FakeAPI(fail_first=[429], token=\"k\")\n    r = send(api, \"POST\", f\"{BASE}/books\", sleep=waits.append, headers={\"Authorization\": \"Bearer k\"}, body=b'{\"title\": \"A\", \"author\": \"B\"}')\n    assert r.status == 201 and waits == [1.0]\n    waits.clear()\n    api = FakeAPI(fail_first=[TimeoutError] * 5)\n    try:\n        send(api, \"GET\", f\"{BASE}/books/1\", attempts=3, base=0.5, sleep=waits.append)\n        assert False\n    except TimeoutError:\n        assert waits == [0.5, 1.0] and len(api.log) == 3")

### Exercise: A small client class

Write a class `BooksClient(api, env, timeout=5)`: it reads its token from `env["BOOKS_TOKEN"]` **when created** (raising `RuntimeError` if it's missing), and keeps default headers (`Accept: application/json`, `Authorization: Bearer TOKEN`, `User-Agent: atlas-books/1.0`). Methods: `get(path, **params)` and `delete(path)` send requests to `BASE + path` (with `params` encoded as a query string) using the default headers and the timeout, and return the response after `raise_for_status()`; `book(book_id)` returns one book's dict; `remove(book_id)` deletes it and returns `True`. Also define `__repr__` as `BooksClient(token=***)`, so the token never appears in logs.

In [ ]:
from urllib.parse import urlencode
from fakeapi import BASE

class BooksClient:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_client():\n    \"defaults in one place, secrets kept out of sight\"\n    from fakeapi import FakeAPI, HTTPError\n    api = FakeAPI(token=\"secret-7\")\n    client = BooksClient(api, {\"BOOKS_TOKEN\": \"secret-7\"})\n    assert client.book(4)[\"title\"] == \"Middlemarch\"\n    assert client.get(\"/books\", page=2).json()[\"page\"] == 2 and api.log[-1][1].endswith(\"/books?page=2\")\n    assert client.remove(4) is True\n    try:\n        client.book(4)\n        assert False\n    except HTTPError as e:\n        assert e.response.status == 404\n    assert repr(client) == \"BooksClient(token=***)\" and \"secret-7\" not in repr(client) and \"secret-7\" not in str(vars(client).keys())\n    try:\n        BooksClient(api, {})\n        assert False\n    except RuntimeError:\n        pass")

### Bug hunt: An error read as data

The client decodes the body without checking the status, so the `{"error": "not found"}` of a 404 is treated as a book, and this crashes with `KeyError: 'title'`. Check the status first, and print `no such book` for a 404, so the program prints `Emma` then `no such book`.

In [ ]:
from fakeapi import FakeAPI, BASE

api = FakeAPI()
for book_id in (1, 99):
    book = api("GET", f"{BASE}/books/{book_id}", timeout=5).json()
    print(book["title"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"status before body\"\n    expect_output(\"Emma\\nno such book\")")

### Bug hunt: A request with no timeout

The server is slow (each request takes 30 seconds), and the client passes no timeout, so in a real program it would wait as long as the server likes; this fake server refuses to pretend and raises instead. Pass a timeout of 5 seconds and catch the `TimeoutError`, so this prints `gave up after 5 seconds`.

In [ ]:
from fakeapi import FakeAPI, BASE

slow = FakeAPI(delay=30)

def fetch(url):
    if True:
        raise RuntimeError("no timeout: this would wait forever")
    return slow("GET", url)

print(fetch(f"{BASE}/books/1"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"always a timeout\"\n    expect_output(\"gave up after 5 seconds\")")

### Bug hunt: The last page that was never fetched

The loop stops when `page` reaches `pages`, before fetching that last page, so book 7 is missed and this prints `6 books`. Fetch the last page too, so it prints `7 books`.

In [ ]:
from fakeapi import FakeAPI, BASE

api = FakeAPI(page_size=3)
books, page, pages = [], 1, None
while pages is None or page < pages:
    data = api("GET", f"{BASE}/books?page={page}", timeout=5).json()
    books += data["items"]
    pages = data["pages"]
    page += 1
    if page == pages:
        break
print(len(books), "books")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every page\"\n    expect_output(\"7 books\")")

### Puzzle: Calling an API with requests

Arrange the lines (for your own computer, with `requests`) that fetch page 2 of an API's books with a token from the environment. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the requests argument that encodes a dict as a JSON body
q = ___   # True or False: requests has a default timeout
r = ___   # the waits, in seconds, for retries 1 to 4 with base 0.5 and doubling, as a list
s = ___   # the header a 429 response may carry, saying how long to wait
t = ___   # True or False: a POST that timed out is always safe to resend

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"json\" and q is False and r == [0.5, 1.0, 2.0, 4.0], \"p, q and r\"\n    assert s == \"Retry-After\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a polite crawler

Write `crawl(api, ids, rate, clock, sleep)`: fetch `GET {BASE}/books/{id}` for each id, in order, and return a dict of id to title (missing books map to `None`). Be polite: pace the requests to an average of `rate` per second, allowing short bursts, with a **token bucket**: the bucket holds at most `rate` tokens, starts full, refills continuously at `rate` tokens per second (measured with `clock()`), and each request takes one token; when it's empty, `sleep()` exactly long enough for one token. Also handle `429`: wait the `Retry-After` seconds, then try the same id again (counting it as another request), at most 3 times per id before raising `RuntimeError(f"rate limited on {id}")`. `sleep(seconds)` must advance whatever `clock()` reads (the checks pass a fake pair).

In [ ]:
from fakeapi import BASE

class TokenBucket:
    def __init__(self, rate, clock):
        ___

def crawl(api, ids, rate, clock, sleep):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_crawl():\n    \"a token bucket, and 429s\"\n    from fakeapi import FakeAPI\n    now = [0.0]\n    sleeps = []\n    def clock():\n        return now[0]\n    def sleep(seconds):\n        sleeps.append(round(seconds, 3))\n        now[0] += seconds\n    api = FakeAPI()\n    sent_at = []\n    def timed(method, url, headers=None, body=None, timeout=None):\n        sent_at.append(round(now[0], 3))\n        return api(method, url, headers, body, timeout)\n    result = crawl(timed, [1, 2, 3, 99, 5], rate=2, clock=clock, sleep=sleep)\n    assert result == {1: \"Emma\", 2: \"Persuasion\", 3: \"Bleak House\", 99: None, 5: \"Dracula\"}\n    assert sent_at == [0.0, 0.0, 0.5, 1.0, 1.5], sent_at\n    now[0] = 0.0\n    sleeps.clear()\n    limited = FakeAPI(fail_first=[429, 429])\n    assert crawl(limited, [4], rate=10, clock=clock, sleep=sleep) == {4: \"Middlemarch\"} and sleeps == [1.0, 1.0]\n    try:\n        crawl(FakeAPI(fail_first=[429] * 5), [1], rate=10, clock=clock, sleep=sleep)\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"rate limited on 1\"")

## Warm-up

`fakeapi.py` (open it above) is a small pretend book API, running inside the page, because the browser sandbox can't reach the real internet. It behaves like a real one: JSON bodies, status codes, authentication, pagination, and (when you ask it to) failures and rate limits. You call it with a method, a URL, headers, a body and a timeout, exactly as a real HTTP library would send them, and get back a `Response` with `.status`, `.headers`, `.json()` and `.raise_for_status()`, the same names the `requests` library uses.

## Requests and JSON

On your own computer you'd use **`requests`** (or the newer **`httpx`**, which has the same interface and also supports async):

```python
import requests

response = requests.get("https://api.example.com/books", params={"page": 2}, timeout=5)
response.raise_for_status()
for book in response.json()["items"]:
    print(book["title"])

created = requests.post("https://api.example.com/books",
                        json={"title": "Emma", "author": "Austen"},
                        headers={"Authorization": f"Bearer {token}"}, timeout=5)
print(created.status_code, created.headers["Location"])
```

- `params=` builds the query string (encoded properly, Day 1); `json=` encodes a body as JSON and sets `Content-Type: application/json`; `data=` sends a form.
- `response.json()` decodes the body; check the status **first**, because an error response is JSON too, just not the JSON you expected.
- **`requests.Session()`** (or `httpx.Client()`) keeps settings (base headers, authentication) and **reuses connections** (keep-alive), which is much faster for many requests to one server.
- APIs usually version their URLs (`/v2/books`) or headers, so they can change without breaking old clients.

Designing a client so the **transport** (the function that actually sends the request) can be swapped is exactly yesterday's dependency injection: in tests (and in this browser), you pass a fake; in production, a real session.

## Errors and authentication

Treat the status code as part of the answer:

- `2xx`: use the body. `201 Created` usually has a `Location` header; `204 No Content` has no body to decode.
- `404`: often an expected answer ("no such book"), returned as `None` rather than an error.
- `400` and `422`: your request was wrong; the body usually explains why. Retrying won't help.
- `401`: authenticate (or your token expired); `403`: you're not allowed.
- `429` and `5xx`: the server's problem or a temporary condition: maybe retry later (below).

**Authentication** for APIs is usually a **token**: an API key or an OAuth access token, sent in a header (`Authorization: Bearer TOKEN`, or a custom `X-API-Key`). Tokens are secrets: read them from the **environment** or a secrets store (`os.environ["BOOKS_TOKEN"]`), never write them in code or commit them to a repository, and don't log them (Week 79). Send them only over HTTPS.

## Pagination

APIs don't return a million records in one response; they return **pages**:

- **Page numbers** (`?page=2&per_page=50`), with the response saying how many pages there are. Simple, but if records are added while you're reading, you can see one twice or miss one.
- **Cursors** (`?after=1234`, or an opaque `next` token in the response): "give me the records after this one". Stable while data changes, and efficient for the server. Keep requesting with the last `next` until it's `null`.
- Some APIs put the next page's URL in a **`Link`** header (`<https://...?page=3>; rel="next"`).

A **generator** (Week 59) is the natural shape for a paginated client: it fetches a page only when the caller has used up the previous one, so a caller who needs the first ten records never fetches the rest.

## Timeouts, retries and rate limits

Networks fail, and a client must expect it:

- **Always set a timeout.** Without one, a request to a server that never answers waits **forever**, and so does your program. `requests` has no default timeout at all; set one on every call (`timeout=5`, or separate connect and read timeouts).
- **Retry only what's safe**: idempotent methods (Day 2) after a network error or a `502`, `503` or `504`. A `POST` that timed out may have succeeded on the server; retrying it could create two orders, unless the API supports an **idempotency key** (a unique header value the server uses to recognise a repeat).
- **Back off exponentially**: wait 1 second, then 2, then 4..., so a struggling server gets room to recover, and add random **jitter** so a thousand clients don't all retry at the same instant. Give up after a few attempts.
- **Rate limits**: APIs cap how many requests you may make (say, 100 a minute). Over the limit, they answer **`429 Too Many Requests`**, often with a **`Retry-After`** header (in seconds): wait at least that long. Better still, pace yourself so you never hit it.

The `tenacity` library and urllib3's `Retry` (used by `requests` adapters) implement all of this; it's worth writing once yourself to know exactly what they do.

## Practice

All of these use `fakeapi.py`'s `FakeAPI` as the transport, passed in as `api`. Every request must have a timeout.

## On your own computer

Call a real public API that needs no key:

```bash
pip install requests
```

```python
import requests

r = requests.get("https://api.github.com/repos/python/cpython", timeout=5)
r.raise_for_status()
data = r.json()
print(data["full_name"], data["stargazers_count"], r.headers.get("X-RateLimit-Remaining"))
```

GitHub's API tells you your remaining rate limit in a header on every response. Then try `httpx` (`pip install httpx`), whose `httpx.get`, `httpx.Client` and `response.raise_for_status()` work the same way, and which can also make many requests concurrently with `async`.

## Quiz

## Daily challenge

## Before you go

Pick a public API you'd like to use (weather, transport, books, museums) and read its documentation for authentication, pagination and rate limits. Tomorrow: **sockets**, the layer underneath every HTTP library: connections, bytes, and the framing that turns a stream into messages.